In [8]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Olist-Silver") \
    .master("local[*]") \
    .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)


In [9]:
base_path = "/Users/sarah/Desktop/brief/BigData"

orders = spark.read.parquet(f"{base_path}/data/1_bronze/orders")
payments = spark.read.parquet(f"{base_path}/data/1_bronze/payments")
reviews = spark.read.parquet(f"{base_path}/data/1_bronze/reviews")

print(f"orders : {orders.count()} rows")
print(f"payments : {payments.count()} rows")
print(f"reviews : {reviews.count()} rows")


orders : 99441 rows
payments : 103886 rows
reviews : 99224 rows


In [10]:
for name, df in [("orders", orders), ("payments", payments), ("reviews", reviews)]:
    print(f"\n{name} :")
    for col in df.columns:
        null_count = df.filter(df[col].isNull()).count()
        if null_count > 0:
            print(f"  {col} : {null_count} nulls")


orders :
  order_approved_at : 160 nulls
  order_delivered_carrier_date : 1783 nulls
  order_delivered_customer_date : 2965 nulls

payments :

reviews :
  review_comment_title : 87656 nulls
  review_comment_message : 58247 nulls


In [19]:
from pyspark.sql.functions import col

orders.filter(
    col("order_approved_at").isNull() 
).show(truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|00b1cb0320190ca0daa2c88b35206009|3532ba38a3fd242259a514ac2b6ae6b6|canceled    |2018-08-28 15:26:39     |NULL             |NULL                        |NULL                         |2018-09-12 00:00:00          |
|ed3efbd3a87bea76c2812c66a0b32219|191984a8ba4cbb2145acb4fe35b69664|canceled    |2018-09-20 13:54:16     |NULL             |NULL                     

In [21]:
# Check incoherent delivered orders without approval date
incoherent = orders.filter(
    (col("order_status") == "delivered") & 
    (col("order_approved_at").isNull())
)
print(f"Delivered orders without approval date : {incoherent.count()}")

Delivered orders without approval date : 14


In [23]:
# Delivered without customer delivery date
print(f"Delivered without customer delivery date : {orders.filter((col('order_status') == 'delivered') & col('order_delivered_customer_date').isNull()).count()}")

# Delivered without carrier date
print(f"Delivered without carrier date : {orders.filter((col('order_status') == 'delivered') & col('order_delivered_carrier_date').isNull()).count()}")

# Delivery date before purchase date
print(f"Delivery before purchase : {orders.filter(col('order_delivered_customer_date') < col('order_purchase_timestamp')).count()}")

# Approval date before purchase date
print(f"Approval before purchase : {orders.filter(col('order_approved_at') < col('order_purchase_timestamp')).count()}")

# Estimated delivery before purchase date
print(f"Estimated delivery before purchase : {orders.filter(col('order_estimated_delivery_date') < col('order_purchase_timestamp')).count()}")

Delivered without customer delivery date : 8
Delivered without carrier date : 2
Delivery before purchase : 0
Approval before purchase : 1
Estimated delivery before purchase : 0


In [ ]:

for name, df in [("orders", orders), ("payments", payments), ("reviews", reviews)]:
    total = df.count()
    distinct = df.distinct().count()
    duplicates = total - distinct
    print(f"{name} : {duplicates} duplicates")

orders : 0 duplicates
payments : 0 duplicates
reviews : 0 duplicates


In [24]:
# For each order_status, count nulls per date column
from pyspark.sql.functions import sum as spark_sum, when

orders.groupBy("order_status").agg(
    spark_sum(when(col("order_approved_at").isNull(), 1).otherwise(0)).alias("null_approved"),
    spark_sum(when(col("order_delivered_carrier_date").isNull(), 1).otherwise(0)).alias("null_carrier"),
    spark_sum(when(col("order_delivered_customer_date").isNull(), 1).otherwise(0)).alias("null_customer")
).orderBy("order_status").show()

+------------+-------------+------------+-------------+
|order_status|null_approved|null_carrier|null_customer|
+------------+-------------+------------+-------------+
|    approved|            0|           2|            2|
|    canceled|          141|         550|          619|
|     created|            5|           5|            5|
|   delivered|           14|           2|            8|
|    invoiced|            0|         314|          314|
|  processing|            0|         301|          301|
|     shipped|            0|           0|         1107|
| unavailable|            0|         609|          609|
+------------+-------------+------------+-------------+



In [ ]:
from pyspark.sql.functions import col


orders_silver = orders.dropDuplicates()


orders_silver.groupBy("order_status").count().orderBy("count", ascending=False).show()


print(f"order_approved_at nulls : {orders_silver.filter(col('order_approved_at').isNull()).count()}")
print(f"order_delivered_carrier_date nulls : {orders_silver.filter(col('order_delivered_carrier_date').isNull()).count()}")
print(f"order_delivered_customer_date nulls : {orders_silver.filter(col('order_delivered_customer_date').isNull()).count()}")

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+

order_approved_at nulls : 160
order_delivered_carrier_date nulls : 1783
order_delivered_customer_date nulls : 2965


26/06/18 12:29:35 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 184038 ms exceeds timeout 120000 ms
26/06/18 12:29:35 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/18 12:29:41 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [26]:

payments_silver = payments.dropDuplicates()


payments_silver.groupBy("payment_type").count().orderBy("count", ascending=False).show()


print(f"negative values : {payments_silver.filter(col('payment_value') <= 0).count()}")

+------------+-----+
|payment_type|count|
+------------+-----+
| credit_card|76795|
|      boleto|19784|
|     voucher| 5775|
|  debit_card| 1529|
| not_defined|    3|
+------------+-----+

negative values : 9


In [27]:
# REVIEWS
reviews_silver = reviews.dropDuplicates() \
    .filter(col("order_id").isNotNull()) \
    .filter(col("review_score").isNotNull())

print("=== REVIEW SCORE VALUES ===")
reviews_silver.groupBy("review_score").count().orderBy("review_score").show()

=== REVIEW SCORE VALUES ===
+------------+-----+
|review_score|count|
+------------+-----+
|           1|11424|
|           2| 3151|
|           3| 8179|
|           4|19142|
|           5|57328|
+------------+-----+



In [28]:
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [29]:
orders_silver.filter(
    col("order_approved_at") < col("order_purchase_timestamp")
).count()

1

In [30]:
orders_silver.filter(
    col("order_delivered_customer_date") < col("order_purchase_timestamp")
).count()

0

In [31]:
orders_silver.filter(
    col("order_approved_at") < col("order_purchase_timestamp")
).show(truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|32b8249f810186f4cd9179daabc1f1dd|6c05f12cb6a66c47774242ee192825c8|delivered   |2018-03-25 03:56:02     |2018-03-25 03:10:20|2018-03-28 21:22:46         |2018-04-01 12:36:43          |2018-04-06 00:00:00          |
+--------------------------------+--------------------------------+------------+------------------------+-------------------+---------------